# Sequential Bayesian Optimization: End-to-End Workflow

このNotebookはrobotorchanの最小例ではなく、**1つのBO campaignを評価予算の開始から終了まで運用する基準workflow**です。

```text
initial observations
      ↓
fit surrogate
      ↓
inspect posterior
      ↓
construct acquisition
      ↓
optimize acquisition
      ↓
candidate
      ↓
expensive evaluation
      ↓
append observation → repeat until budget exhausted
      ↓
report incumbent and history
```

ここではcontinuous single-objective maximizationを使い、複雑なvariantは末尾から専用Notebookへ接続します。

## 1. Imports and campaign configuration

In [ ]:
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.fit import fit_gpytorch_mll
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import SingleTaskGP
from robotorchan.optim import optimize_acqf

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

N_INITIAL = 6
N_BO_STEPS = 5
bounds = torch.tensor([[0.0], [1.0]])

## 2. Expensive black-box objective

実務ではこの関数呼び出しが実験、シミュレーション、製造評価などに置き換わります。BO loopはこの評価責務をsurrogateやacquisitionへ隠しません。

In [ ]:
def expensive_objective(X: torch.Tensor) -> torch.Tensor:
    return (
        0.75 * torch.sin(3.0 * torch.pi * X)
        + 0.30 * torch.cos(7.0 * torch.pi * X)
        + 0.35 * X
    )


initial_X = torch.linspace(0.05, 0.95, N_INITIAL).unsqueeze(-1)
initial_Y = expensive_objective(initial_X)
train_X = initial_X.clone()
train_Y = initial_Y.clone()

print("initial evaluations:", train_X.shape[0])
print("initial best:", train_Y.max().item())

## 3. One complete BO decision

各decisionではmodelを現在のcompleted observationsから作り直します。

In [ ]:
model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(model.make_mll())
model.eval()

grid = torch.linspace(0.0, 1.0, 101).unsqueeze(-1)
posterior = model.posterior(grid)
print("posterior mean shape:", posterior.mean.shape)

sampler = SobolQMCNormalSampler(torch.Size([128]), seed=101)
acquisition = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
    sampler=sampler,
)
candidate, acq_value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    optimizer="botorch",
    num_restarts=4,
    raw_samples=64,
)
candidate = candidate.detach()
observed_Y = expensive_objective(candidate)

print("candidate:", candidate.item())
print("acquisition value:", acq_value.item())
print("observed value:", observed_Y.item())

## 4. Commit the completed observation

candidateを選んだだけではtraining dataは更新しません。高価な評価が完了して`Y`が得られた時点でcompleted observationとして追加します。

In [ ]:
train_X = torch.cat([train_X, candidate], dim=0)
train_Y = torch.cat([train_Y, observed_Y], dim=0)

assert train_X.shape[0] == N_INITIAL + 1
print("completed evaluations:", train_X.shape[0])

## 5. Budgeted sequential BO loop

ここから残りの評価予算を消費します。履歴にはcandidate、観測値、incumbentを保存し、BO campaign終了後に意思決定履歴を追跡できるようにします。

In [ ]:
history = []
for step in range(1, N_BO_STEPS + 1):
    model = SingleTaskGP(train_X, train_Y)
    fit_gpytorch_mll(model.make_mll())
    model.eval()

    acquisition = qLogExpectedImprovement(
        model=model,
        best_f=train_Y.max(),
        sampler=SobolQMCNormalSampler(
            torch.Size([128]),
            seed=200 + step,
        ),
    )
    candidate, acq_value = optimize_acqf(
        acq_function=acquisition,
        bounds=bounds,
        q=1,
        optimizer="botorch",
        num_restarts=4,
        raw_samples=64,
    )
    candidate = candidate.detach()
    observed_Y = expensive_objective(candidate)
    train_X = torch.cat([train_X, candidate], dim=0)
    train_Y = torch.cat([train_Y, observed_Y], dim=0)

    incumbent_index = train_Y.squeeze(-1).argmax()
    history.append(
        {
            "step": step,
            "candidate": float(candidate.item()),
            "observed": float(observed_Y.item()),
            "incumbent_x": float(train_X[incumbent_index].item()),
            "incumbent_y": float(train_Y[incumbent_index].item()),
            "acquisition": float(acq_value.item()),
        }
    )

for record in history:
    print(record)

## 6. Campaign result and stopping condition

この例の停止条件は固定評価予算です。実務では時間、コスト、改善量、uncertainty、feasibilityなどを停止条件にできますが、acquisition valueだけを普遍的な停止閾値とはみなしません。

In [ ]:
best_index = train_Y.squeeze(-1).argmax()
best_X = train_X[best_index]
best_Y = train_Y[best_index]
total_evaluations = train_X.shape[0]
expected_evaluations = N_INITIAL + 1 + N_BO_STEPS

assert total_evaluations == expected_evaluations
print("total completed evaluations:", total_evaluations)
print("final incumbent X:", best_X.item())
print("final incumbent Y:", best_Y.item())

## 7. What belongs to campaign state?

このNotebookでは最小限としてcompleted `train_X/train_Y`と履歴を保持しました。実運用では次も明示的に保存する価値があります。

- objective direction and transforms
- search-space definition and candidate constraints
- random seed / model and acquisition configuration
- completed, pending, failed, and cancelled evaluations
- cost / fidelity / task metadata
- incumbent definition
- stopping reason
- model/version provenance

これらはGP state_dictだけでは復元できないBO campaignの状態です。

## 8. Cross-layer responsibility map

| Layer | This workflow |
| --- | --- |
| Data | completed observations |
| Surrogate | `SingleTaskGP` |
| Fit | `make_mll()` + BoTorch fitting |
| Posterior | model uncertainty over candidate space |
| Sampling | Sobol QMC for MC acquisition |
| Acquisition | qLogEI |
| Initialization | optimizer-owned raw samples / restarts |
| Optimization | public `robotorchan.optim.optimize_acqf` |
| Candidate | nominal next experiment |
| Evaluation | external expensive objective |
| Update | append only completed observation |
| Stop | explicit evaluation budget |

## 9. Choose the workflow that matches the problem

このsequential workflowを全問題へ無理に拡張しません。

| Problem structure | Representative executable notebook |
| --- | --- |
| minimum SingleTaskGP BO | `01_single_task_gp.ipynb` |
| mixed continuous/categorical | `02_mixed_single_task_gp.ipynb` |
| multi-fidelity / cost-aware | `03_multi_fidelity_gp.ipynb` |
| MultiTask / Kronecker | `04_multitask_gp.ipynb` |
| high-dimensional reduction | `19_reduced_gp.ipynb` |
| robust observation surrogate | `10_robust_gp.ipynb` |
| multi-objective acquisition | `27_multiobjective_acquisition.ipynb` |
| regression active learning | `28_active_learning_acquisition.ipynb` |
| batch / async / fantasy | `32_batch_async_fantasization.ipynb` |
| TuRBO / trust region | `34_turbo_trust_region.ipynb` |
| execution input perturbation | `35_robust_input_perturbation.ipynb` |

## 10. Production boundary

Notebookはworkflow semanticsを示すExecutable Documentationです。実験キュー、永続DB、retry policy、worker scheduling、認証、監査ログなどのproduction orchestrationをNotebook helperへ隠しません。

同様に、モデル・acquisition・optimizerの選択を巨大な`run_bo()` helperへまとめないことで、各cross-layer contractを読み取れる状態に保ちます。

## 11. Related documentation

- `docs/theory/01_bayesian_optimization.md`
- `docs/development/notebook-executable-documentation-architecture.md`
- `docs/optimization/acquisition-integration.md`
- `docs/development/notebook-authoring-standard.md`

このNotebookはsequential single-objective BO campaignの基準点であり、各専門workflowは同じlifecycleの必要な層だけを置き換えます。